# Pagination: Page Numbers, Cursors, Links, Watermarks

## What you'll practice

The mock API exposes the three common pagination styles plus an incremental feed: `/customers` (page/page_size), `/orders` (cursor), `/products` (`next` link)
and `/events?since=` (watermark). Generators are the natural fit: callers iterate records and pages are fetched lazily.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/apis/test_07_03_pagination.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/07_apis/03_pagination.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Page-number pagination

**Implement:** `fetch_all_customers(session, base_url: str, page_size: int = 50) -> list[dict]`

Request `/customers?page=N&page_size=M` starting at 1 until `page == total_pages`. Return all records. Use `timeout=10`, `raise_for_status()`.

**Input:** `session, base_url: str, page_size: int = 50`

**Expected output:** `list[dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_fetch_all_customers` in `tests/apis/test_07_03_pagination.py`.

**Check:** `pytest tests/apis/test_07_03_pagination.py -k test_fetch_all_customers`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Cursor pagination as a generator

**Implement:** `iter_orders(session, base_url: str, limit: int = 100) -> Iterator[dict]`

Yield orders from `/orders?limit=L&cursor=C`, passing `next_cursor` from each response (omit `cursor` on the first call), until `next_cursor` is `None`. Pages must be fetched lazily.

**Input:** `session, base_url: str, limit: int = 100`

**Expected output:** `Iterator[dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_iter_orders_lazy` in `tests/apis/test_07_03_pagination.py`.

**Check:** `pytest tests/apis/test_07_03_pagination.py -k test_iter_orders_lazy`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Follow next links

**Implement:** `iter_products(session, url: str, limit: int = 50) -> Iterator[dict]`

Start at `url` with `params={"limit": limit}`; afterwards follow the absolute `next` URL from each response **as-is** (it already contains the query). Stop when `next` is `None`. Guard against a server returning the same `next` twice (raise `RuntimeError`).

**Input:** `session, url: str, limit: int = 50`

**Expected output:** `Iterator[dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_iter_products, test_iter_products_loop_detection` in `tests/apis/test_07_03_pagination.py`.

**Check:** `pytest tests/apis/test_07_03_pagination.py -k "test_iter_products or test_iter_products_loop_detection"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Incremental pull with a watermark

**Implement:** `pull_events(session, base_url: str, since: str, limit: int = 100) -> tuple[list[dict], str]`

Call `/events?since=W&limit=L` repeatedly, moving `W` to the response's `max_updated_at` until a response has no events. Return `(events, new_watermark)` where events are
deduplicated by `event_id` (first seen wins) and `new_watermark` is the max `updated_at` seen, or the original `since` if nothing new. This is how incremental API
ingestion works: persist the watermark after a successful load and pass it next run.

Then run it on the real mock feed (`MockSession()`, `since=""`) and compare `len(events)` with the number of distinct event ids in `session.data["events"]`.
You'll probably lose one. Work out why before reading the solution's explanation.

**Input:** `session, base_url: str, since: str, limit: int = 100`

**Expected output:** `tuple[list[dict], str]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_pull_events, test_pull_events_real_feed` in `tests/apis/test_07_03_pagination.py`.

**Check:** `pytest tests/apis/test_07_03_pagination.py -k "test_pull_events or test_pull_events_real_feed"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · One paginator, three strategies · 🧩 Mini-project

**Implement:** `paginate(session, url: str, style: str, items_key: str, page_size: int = 100, max_pages: int | None = None) -> Iterator[dict]`

A reusable generator for any API: `style` is `"page"` (params `page`, `page_size`; stop when `page >= total_pages` or a page is empty), `"cursor"` (params `limit`, `cursor`;
uses `next_cursor`) or `"link"` (param `limit` on the first call; then follow `next`). `items_key` says where records live (`"data"`, `"items"`, `"results"`).
`max_pages` caps requests (safety valve for runaway pagination) — stop silently when reached. Unknown style → `ValueError` **at call time** (before iteration — hint: validate in a
normal function that returns a generator). Record `paginate.last_pages` = pages fetched by the most recent completed or stopped iteration is NOT required.

**Input:** `session, url: str, style: str, items_key: str, page_size: int = 100, max_pages: int | None = None`

**Expected output:** `Iterator[dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_paginate_all_styles, test_paginate_max_pages_and_validation` in `tests/apis/test_07_03_pagination.py`.

**Check:** `pytest tests/apis/test_07_03_pagination.py -k "test_paginate_all_styles or test_paginate_max_pages_and_validation"`

**YOUR CODE ↓**